In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=nz4BiRMyLyJwt0UUNR1XhsEgzSNJsP&access_type=offline&code_challenge=2k0AvXm7H4SI0tq2NpoGcU1r0qwvUFTzgMGhPc5w1ms&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/02 14:19:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder="gs://open-targets-data-releases/25.06/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

# Diseases

In [ ]:
#qst=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_studies").select("studyId").distinct().cache()
#qst.count()

15715

In [4]:
qd_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/gentropy_paper/qualifying_credible_sets").select("studyLocusId").cache()
qd_cs.count()

70618

In [5]:
df=sl.df.join(qd_cs,on="studyLocusId",how="inner").cache()
df.count()

25/09/02 14:20:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


70618

In [6]:
df.groupBy("fineMappingMethod").count().show()

+-----------------+-----+
|fineMappingMethod|count|
+-----------------+-----+
|            SuSie|15075|
|        SuSiE-inf|27561|
|             PICS|27982|
+-----------------+-----+



In [7]:
df=df.filter((f.col("fineMappingMethod") == "SuSiE-inf")|(f.col("fineMappingMethod") == "SuSie")).cache()
df.count()

42636

In [8]:
fm_max=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/max_feature_per_CS.parquet")

In [9]:
df=df.join(fm_max,on="studyLocusId",how="inner").cache()
df.count()

42081

In [10]:
df.filter(f.col("region").isNull()).count()

0

In [11]:
df.select("region").distinct().count()

24558

In [12]:
df.show(1)

+--------------------+---------+-------------+----------+--------+-----------------+------------+------------------+-----------------+--------------+--------------+-------------------------------+-------------+-------------------+--------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+--------------------+--------------------+----------+---------------------+---------------------+----------------------------------+-------------------------+--------------------------------------+-------------------+--------------------------------+---------------+----------------------------+--------------------+---------------------------------+------------------+-------------------------------+--------------+--------------------+---------------------------------+------------------+-------------------------------+---------------------+--------------------+---------------------------------+------------------+---------------

In [13]:
def get_feature_numbers(dff: DataFrame) -> None:
    from pyspark.sql.window import Window

    # Define the window partitioned by region and ordered by p-value (ascending)
    window = Window.partitionBy("region","studyId").orderBy(
        (f.col("pValueMantissa") * (10 ** f.col("pValueExponent"))).asc()
    )

    # Add the rank column
    df1 = dff.withColumn(
        "region_rank",
        f.row_number().over(window)
    )
    print("total number of regions:", df1.select("region").distinct().count())
    df2=df1.filter(f.col("region_rank") > 1).select("region").distinct()
    print("total number of regions with secondary signals:", df2.count())
    df1=df1.join(df2, on="region", how="inner")

    prime_count=df1.filter(f.col("region_rank") == 1).count()
    secondary_count=df1.filter(f.col("region_rank") > 1).count()

    prime_vep_count = df1.filter((f.col("region_rank") == 1)&(f.col("vepMaximum")>=0.66)).count()
    secondary_vep_count = df1.filter((f.col("region_rank") > 1)&(f.col("vepMaximum")>=0.66)).count()

    prime_eqtl_count = df1.filter((f.col("region_rank") == 1)&((f.col("eQtlColocClppMaximum")>=0.01)|(f.col("eQtlColocH4Maximum")>=0.8))).count()
    secondary_eqtl_count = df1.filter((f.col("region_rank") > 1)&((f.col("eQtlColocClppMaximum")>=0.01)|(f.col("eQtlColocH4Maximum")>=0.8))).count()

    prime_pqtl_count = df1.filter((f.col("region_rank") == 1)&((f.col("pQtlColocClppMaximum")>=0.01)|(f.col("pQtlColocH4Maximum")>=0.8))).count()
    secondary_pqtl_count = df1.filter((f.col("region_rank") > 1)&((f.col("pQtlColocClppMaximum")>=0.01)|(f.col("pQtlColocH4Maximum")>=0.8))).count()

    df3 = (df1.filter((f.col("region_rank") == 1)&(f.col("vepMaximum")<0.66)&(f.col("eQtlColocClppMaximum")<0.01)
        &(f.col("eQtlColocH4Maximum")<0.8)&(f.col("pQtlColocClppMaximum")<0.01)&(f.col("pQtlColocH4Maximum")<0.8))).select("region").distinct()
    print("total number of regions where primary signal has no features:", df3.count())
    no_primary_but_secondary_count=df1.join(df3, on="region", how="inner").filter((f.col("vepMaximum")>=0.66)|(f.col("eQtlColocClppMaximum")>=0.01)
        |(f.col("eQtlColocH4Maximum")>=0.8)|(f.col("pQtlColocClppMaximum")>=0.01)|(f.col("pQtlColocH4Maximum")>=0.8))
    print("proportion of regions where primary signal has no features but secondary signal has features:", no_primary_but_secondary_count.count()/df3.count())

    print("total number of prime CSs:", prime_count)
    print("total number of secondary CSs:", secondary_count)
    print("total proportion of prime CSs with vepMaximum >= 0.66:", prime_vep_count/prime_count)
    print("total proportion of secondary CSs with vepMaximum >= 0.66:", secondary_vep_count/secondary_count)
    print("total proportion of prime CSs with eqtls:", prime_eqtl_count/prime_count)
    print("total proportion of secondary CSs with eqtls:", secondary_eqtl_count/secondary_count)
    print("total proportion of prime CSs with pQtls:", prime_pqtl_count/prime_count)
    print("total proportion of secondary CSs with pQtls:", secondary_pqtl_count/secondary_count)


In [14]:
get_feature_numbers(df)

total number of regions: 24558


total number of regions with secondary signals: 6354


total number of regions where primary signal has no features: 2862


proportion of regions where primary signal has no features but secondary signal has features: 0.8151642208245982
total number of prime CSs: 8780
total number of secondary CSs: 11439
total proportion of prime CSs with vepMaximum >= 0.66: 0.17539863325740318
total proportion of secondary CSs with vepMaximum >= 0.66: 0.14642888364367515
total proportion of prime CSs with eqtls: 0.484624145785877
total proportion of secondary CSs with eqtls: 0.42425037153597345
total proportion of prime CSs with pQtls: 0.09772209567198177
total proportion of secondary CSs with pQtls: 0.07168458781362007


In [15]:
repl_cs=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/list_of_gwas_replicated_CSs.parquet")

In [16]:
df1_repl=df.join(repl_cs,on="studyLocusId",how="inner").cache()
df1_repl.count()

12150

In [17]:
get_feature_numbers(df1_repl)

total number of regions: 8470


total number of regions with secondary signals: 1168


total number of regions where primary signal has no features: 591


proportion of regions where primary signal has no features but secondary signal has features: 0.7377326565143824
total number of prime CSs: 1644
total number of secondary CSs: 1839
total proportion of prime CSs with vepMaximum >= 0.66: 0.1745742092457421
total proportion of secondary CSs with vepMaximum >= 0.66: 0.10984230560087004
total proportion of prime CSs with eqtls: 0.3734793187347932
total proportion of secondary CSs with eqtls: 0.33931484502446985
total proportion of prime CSs with pQtls: 0.128345498783455
total proportion of secondary CSs with pQtls: 0.09189777052746058


12444